Imports and install xgboost

In [0]:
%pip install xgboost

Restart Python Kernal after installing xgboost

In [0]:
%restart_python

### Create Table with NP confirmed removed.
ANTI JOIN based on the PTID of matching records to the NP confirmed table

In [0]:
%sql
CREATE OR REPLACE TABLE final_multimodal_adni_living AS
SELECT f.*
FROM final_multimodal_adni f
LEFT ANTI JOIN merged_neuropath_multimodal_adni d
  ON f.PTID = d.PTID

The dataset I downloaded and sent to you is shown in the below cell:

In [0]:
%sql
SELECT *
FROM final_multimodal_adni_living
LIMIT 10;

In [0]:
%sql
SELECT COUNT(*) AS living_count
FROM final_multimodal_adni_living;

In [0]:
df = spark.read.table('workspace.default.final_multimodal_adni_living')
display(df.describe())

Checking total feature count, should be 213

In [0]:
%sql
SELECT COUNT(*) AS num_columns
FROM information_schema.columns
WHERE table_name = 'final_multimodal_adni_living';

Total record count for later missingness check

In [0]:
total_rows = df.count()

Null count for missingness check

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, when

missing_df = df.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
])


Missingness summary

In [0]:
missing_summary = (
    missing_df
    .toPandas()
    .T
    .reset_index()
)

missing_summary.columns = ["column", "missing_count"]
missing_summary["missing_pct"] = missing_summary["missing_count"] / total_rows
display(missing_summary)

In [0]:
missing_summary_sorted = missing_summary.sort_values("missing_pct", ascending=True)
display(missing_summary_sorted)

Now look for any values with near zero variance.

In [0]:
nzv_stats = []

for c in df.columns:
    counts = (
        df.groupBy(c)
          .count()
          .orderBy(col("count").desc())
    )
    
    top = counts.first()
    
    if top is not None:
        nzv_stats.append((
            c,
            counts.count(),           # num of unique values
            top["count"],             # most frequent value count
            top["count"] / total_rows # ratio
        ))


In [0]:
# store as Spark DataFrame and display to see where there are nzv cols
nzv_summary = spark.createDataFrame(
    nzv_stats,
    ["column", "unique_count", "top_count", "top_ratio"]
)

display(nzv_summary.orderBy(col("top_ratio").desc()))


Join these tables together to see which features we should look at.
- Change nzv to pandas, then join on column name

In [0]:
nzv_pd = nzv_summary.toPandas()

In [0]:
feature_check = missing_summary.merge(
    nzv_pd,
    on="column",        # merging using the column name
    how="left"
)

An issue arises where we see that the missing count is so high on some that it is the dominant value. We can take this into account when looking into features that we might want to remove before conducting actual feature selection and engineering techniques like correlation or VIF so make sure we aren't wasting time on bad variables with high missingness.

pet_processing and VSHGTSC are variables with 100% NULL values, which we should almost surely remove entirely

As for other values, we should look to remove any where 

In [0]:
display(
    feature_check.sort_values(
        ["missing_pct", "top_ratio"],
        ascending=[True, True]
    )
)

### Create binary MMSE_Category Label, and new delta table

In [0]:
%sql
CREATE OR REPLACE TABLE multimodal_adni_living_binary AS
  SELECT *,
    CASE 
      WHEN MMSE_Category IN (
        'Mild AD',
        'Moderate AD',
        'Severe AD'
      ) THEN 1
      WHEN MMSE_Category IN (
        'Normal cognition (Fully intact)',
        'Normal cognition / Possible MCI (Borderline)'
      ) THEN 0
      WHEN MMSE_Category = 'Invalid Score' THEN NULL  
      ELSE NULL 
  END AS y_mmse_binary
FROM final_multimodal_adni_living

Check how many null y_mmse_binary occurances

In [0]:
%sql
FROM multimodal_adni_living_binary
SELECT 
  PTID,
  MMSE_Category, 
  y_mmse_binary
WHERE y_mmse_binary IS NULL
LIMIT 100;

## Remove Data Leakage & PET Columns
Removing any columns that might cause a Data Leakage (ID columns), as well as any PET Columns, since we arent using any image data, and just want biomarkers (probably).

### Pull delta table into spark and check class imbalance
Checking how many positive vs negative cases.

In [0]:
from pyspark.sql import functions as F
table_name = "multimodal_adni_living_binary"

df = spark.table(table_name)

# remove rows where label is null/missing
df_model = df.filter(F.col("y_mmse_binary").isNotNull())

# drop data leakage/IDs
id_drop_cols = ["PTID", "MMSE_Category", "image_id", "study_id", "MMSCORE", "WORLDSCORE", "NPISCORE"] # ID Columns, but ALSO MMScore is leakage because it directly makes the MMSE_Category variable, causing it to also be a data leakage.
df_model = df_model.drop(*[c for c in id_drop_cols if c in df_model.columns]) # drop columns if they exist in the dataframe

# drop pet stuff
pet_drop_cols = [
    "pet_visit",
    "pet_date",
    "pet_description",
    "pet_mfr",
    "pet_mfr_model",
    "pet_reconstruct",
    "pet_height",
    "pet_width",
    "pet_n_images",
    "pet_n_frames",
    "pet_pixel_x",
    "pet_pixel_y",
    "pet_thickness",
    "pet_cnts_src",
    "pet_radiopharm",
    "pet_isotope",
    "pet_processing",
    "pet_decay_corr",
    "pet_sctr_corr",
    "pet_atten_corr",
    "pet_rndm_corr",
    "pet_cv_kernel"
]
df_model = df_model.drop(*[c for c in pet_drop_cols if c in df_model.columns]) # again, check above comment


display(df_model.select("y_mmse_binary").groupBy("y_mmse_binary").count())

In [0]:
for col in df_model.columns:
    print(col)

In [0]:
pdf = df_model.toPandas()
pdf.shape, pdf.head()

In [0]:
import pandas as pd
# Separate minority and majority classes
df_1 = pdf[pdf['y_mmse_binary'] == 1]
df_0 = pdf[pdf['y_mmse_binary'] == 0]
 
# Randomly sample from majority class
df_0_sampled = df_0.sample(n=len(df_1), random_state=42)
 
# Combine and shuffle
balanced_df = (
    pd.concat([df_1, df_0_sampled])
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)
 
print(balanced_df['y_mmse_binary'].value_counts())
pdf = balanced_df

In [0]:
print(pdf.head())

Split into X and y for both XGBoost and LogisticRegression

Log will be used much later, scroll down

In [0]:
X = pdf.drop(columns=["y_mmse_binary"])
y = pdf["y_mmse_binary"].astype(int)

Handle string columns in features due to XGBoost limitations using encoding

show shape after (4537 rows, 285 columns) column increase due to encoding categorical columns

In [0]:
import pandas as pd
X_enc = pd.get_dummies(X,drop_first=False, dummy_na=True)
X_enc.shape

Check for IDs, to make sure (first check still included NPID, NPIDSEV, and GDAFRAID)
- Check these variables first before removing, make sure they are actual identifiers
- NPID: Just happens to have ID, but its question 4(D) of the NPI
- NPIDSEV: The severity of the above question
- GDAFRAID: Also not an ID, but the 4th (D) question

In [0]:
# Check for IDs
[c for c in X.columns if "id" in c.lower()]
# Check for Scores
[c for c in X.columns if "score" in c.lower()]
# WORLDSCORE, MMSCORE, NPISCORE removed

Now we will actually work on the training, using the encoded features as the feature-set, and y being the y_mmse_binary.

In [0]:
X = X_enc
y = y.astype(int) # making sure its still integers only.
y.value_counts()

In [0]:
from sklearn.model_selection import train_test_split
# Train/test split, stratified to keep class ratio consistent

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    stratify=y,
    random_state=42, # 42 because its the answer to the universe
    test_size=0.3
)

In [0]:
print(y_train.value_counts())
print()
print(y_test.value_counts())
print()

Handle class imbalances

In [0]:
import numpy as np

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos
scale_pos_weight

## Baseline Model Training (XGBoost)

In [0]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators = 200,
    max_depth = 3,
    learning_rate = 0.1,
    subsample = 0.8,
    colsample_bytree = 0.8,
    eval_metric = "logloss",
    random_state = 42,
    n_jobs = 1
)

model.fit(X_train, y_train)

Eval

### 1st Result - 90.5%/9.5%
*Imbalanced Dataset*
- ROC_AUC: 97%
- PR_AUC: 80%

### 2nd Result - 50%/50%
*Balanced Dataset*
- ROC_AUC: 94.60%
- PR_AUC: 94.09%

In [0]:
from sklearn.metrics import roc_auc_score, average_precision_score

proba = model.predict_proba(X_test)[:, 1]
print("ROC-AUC:", roc_auc_score(y_test, proba))
print("PR-AUC :", average_precision_score(y_test, proba))

### Data Leakage Check
Created due to AUC = 1/0.9999999
- Found and removed MMSCORE, NPISCORE, WORLDSCORE

In [0]:
# Check for label or mmse_category in the final features
print("y_mmse_binary in X?", "y_mmse_binary" in X_train.columns)
print("MMSE_Category in X?", "MMSE_Category" in X_train.columns)

In [0]:
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

# indices should match within each split
print((X_train.index == y_train.index).all(), (X_test.index == y_test.index).all())


In [0]:
import pandas as pd

leak_cols = []
for c in X_train.columns:
    s0 = X_train.loc[y_train==0, c]
    s1 = X_train.loc[y_train==1, c]
    # if ranges don't overlap at all, it's suspicious
    if s0.max() < s1.min() or s1.max() < s0.min():
        leak_cols.append(c)

leak_cols[:50], len(leak_cols)


## Next Steps:
### Step 1:
- Fix Class Imbalance by taking 432 AD and non-AD, rather than deaing with a class imbalance.
- Verify with Cross-Validation after.

### Step 2:
- Check Records themselves, account for records where too many features are missing from an actual record.
- Look at feature selection, SHAP, VIF, Correlation Map

Don't look at Hyperparameter tuning until later.